In [ ]:
#only need to run once everytime you start colab
!unzip asl_dataset.zip -d asl_dataset

Archive:  secret_data.zip
   creating: secret_data/secret_data/
   creating: secret_data/secret_data/a/
  inflating: secret_data/secret_data/a/A-!-0.jpg  
  inflating: secret_data/secret_data/a/A-!-12.jpg  
  inflating: secret_data/secret_data/a/A-!-14.jpg  
  inflating: secret_data/secret_data/a/A-!-5.jpg  
  inflating: secret_data/secret_data/a/A-!-54.jpg  
  inflating: secret_data/secret_data/a/A-!-59.jpg  
  inflating: secret_data/secret_data/a/A-!-67.jpg  
  inflating: secret_data/secret_data/a/A-!-71.jpg  
  inflating: secret_data/secret_data/a/A-!-73.jpg  
  inflating: secret_data/secret_data/a/A-!-82.jpg  
   creating: secret_data/secret_data/b/
  inflating: secret_data/secret_data/b/B-!-0.jpg  
  inflating: secret_data/secret_data/b/B-!-11.jpg  
  inflating: secret_data/secret_data/b/B-!-116.jpg  
  inflating: secret_data/secret_data/b/B-!-30.jpg  
  inflating: secret_data/secret_data/b/B-!-36.jpg  
  inflating: secret_data/secret_data/b/B-!-49.jpg  
  inflating: secret_data/s

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torch.utils.data import Dataset

# Fix: custom wrapper to apply different transforms to train vs val
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        return self.transform(x), y

    def __len__(self):
        return len(self.subset)

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.RandomRotation(15),  # removed RandomHorizontalFlip
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load dataset WITHOUT any transform first
data_path = 'asl_dataset/asl_dataset'
#dataset = datasets.ImageFolder(root=data_path, transform=None)
dataset = datasets.ImageFolder(root=data_path)

print("Classes found:", dataset.classes)
print("Total images:", len(dataset))

# 80/20 split
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_subset, val_subset = random_split(dataset, [train_size, val_size])

# Apply separate transforms to each subset
train_dataset = TransformSubset(train_subset, train_transform)
val_dataset = TransformSubset(val_subset, val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {train_size}")
print(f"Validation samples: {val_size}")

Classes found: ['a', 'b', 'c', 'd', 'e']
Total images: 49
Training samples: 39
Validation samples: 10


In [ ]:
import torch.nn as nn
class ASLNet(nn.Module):
    def __init__(self):
        super(ASLNet, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        self.fc_layers = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 5)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

model = ASLNet()
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


Total parameters: 32,261


In [ ]:
import torch.optim as optim
from sklearn.metrics import f1_score
# for predicting certain letters print
class_names = dataset.classes

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 60

for epoch in range(epochs):
    model.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = 100 * train_correct / train_total

    model.eval()
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # for printing predicted
            for i in range(min(5, len(labels))):  # print 5 examples
                actual = class_names[labels[i].item()]
                pred = class_names[predicted[i].item()]
                print(f"Actual: {actual} | Predicted: {pred}")

            break  # only show 1 batch so it doesn't spam


    val_acc = 100 * val_correct / val_total
    f1 = f1_score(all_labels, all_preds, average='macro')

    print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}% | F1: {f1:.4f}")



Actual: e | Predicted: e
Actual: a | Predicted: c
Actual: c | Predicted: c
Actual: e | Predicted: d
Actual: d | Predicted: a
Epoch 1/60 | Train Acc: 30.8% | Val Acc: 30.0% | F1: 0.2143
Actual: e | Predicted: e
Actual: a | Predicted: e
Actual: c | Predicted: c
Actual: e | Predicted: d
Actual: d | Predicted: a
Epoch 2/60 | Train Acc: 28.2% | Val Acc: 30.0% | F1: 0.2133
Actual: e | Predicted: e
Actual: a | Predicted: c
Actual: c | Predicted: c
Actual: e | Predicted: d
Actual: d | Predicted: a
Epoch 3/60 | Train Acc: 38.5% | Val Acc: 30.0% | F1: 0.2143
Actual: e | Predicted: c
Actual: a | Predicted: c
Actual: c | Predicted: a
Actual: e | Predicted: d
Actual: d | Predicted: a
Epoch 4/60 | Train Acc: 35.9% | Val Acc: 10.0% | F1: 0.0800
Actual: e | Predicted: c
Actual: a | Predicted: c
Actual: c | Predicted: a
Actual: e | Predicted: d
Actual: d | Predicted: a
Epoch 5/60 | Train Acc: 35.9% | Val Acc: 20.0% | F1: 0.1300
Actual: e | Predicted: c
Actual: a | Predicted: c
Actual: c | Predicted: a
